# 62 — QCA9880: jamming debole (distanza e potenza)

**Corpus:** `monaco/spectral_scans_QCA9880`, con manifest `monaco/inventory_full.csv`. Stessa scheda QCA9880, stesse 8 feature dello spectral scan più `snr`. Si usa solo la larghezza ht20, l'unica con file benigni.
- **Jamming:** 2.260 file in camera schermata, con jammer su un canale noto (25 canali tra 2,4 e 5 GHz) a **20, 40 o 60 cm** e **0, 5 o 10 dBm**.
- **Benigni senza jammer:** 10 file in camera, 10 in laboratorio, 10 in ufficio.
- Ogni file è una scansione di tutti i 25 canali, con circa 460–700 righe per canale.

**Esperimento.**
- Rilevatore sul canale del jammer: Mahalanobis con shrinkage sulle medie per finestra di W righe, addestrato sui benigni in camera **a quel canale**.
- I benigni in camera sono solo 10 file, quindi si usa il **leave-one-file-out**: ogni file benigno è valutato con un modello addestrato sugli altri 9.
- AUROC e rilevamento a soglia zero-falsi-allarmi, in funzione di **distanza × potenza × W**.
- **Controlli:**
  - punteggio fisico semplice: RSSI medio della finestra;
  - canali lontani dal jammer (scorciatoie): AUROC atteso ≈ 0,5;
  - laboratorio e ufficio come benigni fuori distribuzione.

In [ ]:
import os, sys, time, pathlib
REPO = pathlib.Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent: REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np, pandas as pd
from multiprocessing import Pool
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score
from qdetect import passive as P
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 300)
BASE = pathlib.Path(os.environ.get('QCA_BASE', '/home/habes/NotebookProjects/bboi/monaco'))
ROOT = BASE / 'spectral_scans_QCA9880'
N_JOBS = int(os.environ.get('N_JOBS', '16'))
OUT = REPO / 'results' / 'qca9880'; OUT.mkdir(parents=True, exist_ok=True)
inv = pd.read_csv(BASE / 'inventory_full.csv')
inv = inv[(inv.is_processed == True) & (inv.band == 'ht20')].copy()
inv['kind'] = np.where(inv.freq_mhz.notna(), 'jam', 'benign')
inv['banda'] = np.where(inv.freq_mhz < 3000, '2.4', np.where(inv.freq_mhz.notna(), '5', ''))
print(inv.groupby(['kind', 'environment']).size())
WINDOWS = (8, 32, 128)

## 1. Finestre per canale

In [ ]:
def work(row):
    rel, kind = row
    d = pd.read_csv(ROOT / rel)
    mw = 5 if kind == 'benign' else 1
    return [dict(r, full_path=rel) for r in P.channel_windows(d, P.ALL7, WINDOWS, max_windows=mw)]
t0 = time.time()
with Pool(N_JOBS) as pool:
    recs = [r for part in pool.imap_unordered(work, list(zip(inv.full_path, inv.kind)), chunksize=16) for r in part]
F = pd.DataFrame(recs).merge(inv[['full_path', 'kind', 'environment', 'freq_mhz', 'distance_cm', 'jam_power_dbm', 'banda']], on='full_path')
print(f'{inv.shape[0]} file -> {len(F)} finestre in {time.time()-t0:.0f} s')
F.to_pickle(OUT / 'finestre_qca.pkl.gz')

## 2. Rilevamento sul canale del jammer, leave-one-file-out sui benigni in camera

Per ogni canale f e ogni W:
- 10 modelli, ognuno addestrato su 9 file benigni in camera e valutato sul file escluso (negativi) e su tutte le finestre di jamming a f (positivi);
- punteggio dei positivi = media sui 10 modelli.

La **soglia "zero falsi allarmi"** è il punteggio benigno massimo: il rilevamento a quella soglia è una stima prudente.

In [ ]:
fields = {'all7': [f'm_{k}' for k in P.ALL7], 'core': [f'm_{k}' for k in P.CORE]}
ben_files = sorted(F[(F.kind == 'benign') & (F.environment == 'chamber')].full_path.unique())
rows, scores_store = [], {}
t0 = time.time()
for fs_name, cols in fields.items():
    for W in WINDOWS:
        G = F[F.W == W]
        for f in sorted(F[F.kind == 'jam'].freq_mhz.unique()):
            Gc = G[G.channel == f]
            ben = Gc[(Gc.kind == 'benign') & (Gc.environment == 'chamber')]
            jam = Gc[(Gc.kind == 'jam') & (Gc.freq_mhz == f)]
            if ben.full_path.nunique() < 5 or len(jam) == 0:
                continue
            neg, pos = [], np.zeros(len(jam))
            for hold in ben.full_path.unique():
                tr = ben[ben.full_path != hold][cols].to_numpy(float)
                mu, sd = tr.mean(0), tr.std(0) + 1e-6
                lw = LedoitWolf().fit((tr - mu) / sd)
                neg.append(lw.mahalanobis((ben[ben.full_path == hold][cols].to_numpy(float) - mu) / sd))
                pos += lw.mahalanobis((jam[cols].to_numpy(float) - mu) / sd)
            neg = np.concatenate(neg); pos /= ben.full_path.nunique()
            thr = neg.max()
            scores_store[(fs_name, W, f)] = (neg, pos, jam[['distance_cm', 'jam_power_dbm']].to_numpy())
            for (dist, pw), idx in jam.groupby(['distance_cm', 'jam_power_dbm']).indices.items():
                p = pos[idx]; y = np.r_[np.zeros(len(neg)), np.ones(len(p))]
                rssi_auc = roc_auc_score(y, np.r_[ben.m_rssi.to_numpy(), jam.m_rssi.to_numpy()[idx]])
                rows.append(dict(fields=fs_name, W=W, freq=f, banda='2.4' if f < 3000 else '5', distance_cm=dist, power_dbm=pw,
                                 n_neg=len(neg), n_pos=len(p), auroc=roc_auc_score(y, np.r_[neg, p]),
                                 tpr_zero_fp=float((p > thr).mean()), auroc_rssi=rssi_auc))
R = pd.DataFrame(rows); R.to_csv(OUT / 'jamming_debole.csv', index=False)
print(f'{time.time()-t0:.0f} s, {len(R)} righe')

In [ ]:
A = R[R.fields == 'all7']
for W in WINDOWS:
    print(f'\n=== W = {W}: AUROC medio sui canali (righe = distanza, colonne = potenza), per banda ===')
    display(A[A.W == W].pivot_table(index=['banda', 'distance_cm'], columns='power_dbm', values='auroc', aggfunc='mean').round(3))
    print('rilevamento a zero falsi allarmi (media sui canali):')
    display(A[A.W == W].pivot_table(index=['banda', 'distance_cm'], columns='power_dbm', values='tpr_zero_fp', aggfunc='mean').round(3))
print('\n=== canali più difficili (W = 8, AUROC minimo su distanza e potenza) ===')
display(A[A.W == 8].groupby('freq').auroc.min().sort_values().head(8).round(3))
print('\n=== confronto con il solo RSSI medio (AUROC medio, W = 32) ===')
display(A[A.W == 32].pivot_table(index=['banda', 'distance_cm'], columns='power_dbm', values='auroc_rssi', aggfunc='mean').round(3))
print('\n=== senza noise e total_gain_db (core), W = 8 ===')
display(R[(R.fields == 'core') & (R.W == 8)].pivot_table(index=['banda', 'distance_cm'], columns='power_dbm', values='auroc', aggfunc='mean').round(3))

## 3. Controlli: canali lontani dal jammer e benigni fuori distribuzione

In [ ]:
cols = fields['all7']; W = 32; ctrl = []
G = F[F.W == W]
for f in sorted(F[F.kind == 'jam'].freq_mhz.unique()):
    band_ch = [c for c in G.channel.unique() if (c < 3000) == (f < 3000) and abs(c - f) >= (40 if f > 3000 else 30)]
    for c in band_ch[:3]:
        Gc = G[G.channel == c]
        ben = Gc[(Gc.kind == 'benign') & (Gc.environment == 'chamber')]; jam = Gc[(Gc.kind == 'jam') & (Gc.freq_mhz == f)]
        if len(ben) < 10 or len(jam) == 0: continue
        tr = ben[cols].to_numpy(float); mu, sd = tr.mean(0), tr.std(0) + 1e-6; lw = LedoitWolf().fit((tr - mu) / sd)
        s = np.r_[lw.mahalanobis((ben[cols].to_numpy(float) - mu) / sd), lw.mahalanobis((jam[cols].to_numpy(float) - mu) / sd)]
        y = np.r_[np.zeros(len(ben)), np.ones(len(jam))]
        ctrl.append(dict(freq=f, monitored=c, auroc=roc_auc_score(y, s)))
C = pd.DataFrame(ctrl); C.to_csv(OUT / 'controllo_canali_lontani.csv', index=False)
print('AUROC sui canali lontani dal jammer (in-sample per i benigni, quindi ottimista):'); display(C.auroc.describe().round(3))
ood = []
for W in WINDOWS:
    G = F[F.W == W]
    for f in sorted(F[F.kind == 'jam'].freq_mhz.unique()):
        Gc = G[G.channel == f]
        ben = Gc[(Gc.kind == 'benign') & (Gc.environment == 'chamber')]
        if len(ben) < 10: continue
        tr = ben[cols].to_numpy(float); mu, sd = tr.mean(0), tr.std(0) + 1e-6; lw = LedoitWolf().fit((tr - mu) / sd)
        neg_lofo = scores_store[('all7', W, f)][0]; thr = np.quantile(neg_lofo, 0.99)
        for env in ['lab', 'office']:
            o = Gc[(Gc.kind == 'benign') & (Gc.environment == env)]
            if len(o): ood.append(dict(W=W, freq=f, env=env, fpr=float((lw.mahalanobis((o[cols].to_numpy(float) - mu) / sd) > thr).mean())))
O = pd.DataFrame(ood); O.to_csv(OUT / 'ood_lab_office.csv', index=False)
print('Falso allarme su laboratorio/ufficio con soglia = 99° percentile dei benigni in camera:')
display(O.pivot_table(index='env', columns='W', values='fpr', aggfunc='mean').round(3))

## 4. Disegno nella stessa campagna (controllo della sessione)

I 10 file benigni in camera sembrano di un'altra sessione: nella sezione 3 anche i canali lontani dal jammer separano benigni e jamming. Qui per ogni canale f i **negativi** sono le finestre a f dei file della **stessa campagna** con il jammer **nell'altra banda** (a 2,4 GHz se f è a 5 GHz, e viceversa). I **positivi** sono le finestre a f dei file con il jammer proprio su f.
- Split per file 50/25/25; Mahalanobis con shrinkage addestrato sui negativi di training.
- Soglia conforme sui negativi di calibrazione (α = 1% e 0,1%), metriche sui file di test.
- Così le differenze di sessione e di periodo di acquisizione si annullano: resta solo l'effetto del jammer sul canale.

In [ ]:
from qdetect.oneclass import conformal_threshold, fpr_upper95
cols = fields['all7']
J = F[F.kind == 'jam'].copy()
J['split'] = J.full_path.map(dict(zip(J.full_path.unique(), P.file_split(J.full_path.unique().tolist()))))
rows = []
for W in WINDOWS:
    G = J[J.W == W]
    for f in sorted(J.freq_mhz.unique()):
        Gc = G[G.channel == f]
        negs = Gc[(Gc.freq_mhz < 3000) != (f < 3000)]          # jammer nell'altra banda
        poss = Gc[Gc.freq_mhz == f]
        tr, ca, te = (negs[negs.split == s] for s in ('train', 'cal', 'test'))
        if len(tr) < 50 or len(ca) < 50: continue
        X = tr[cols].to_numpy(float); mu, sd = X.mean(0), X.std(0) + 1e-6
        lw = LedoitWolf().fit((X - mu) / sd); sc = lambda d: lw.mahalanobis((d[cols].to_numpy(float) - mu) / sd)
        s_cal, s_neg = sc(ca), sc(te)
        for (dist, pw), p in poss[poss.split == 'test'].groupby(['distance_cm', 'jam_power_dbm']):
            s_pos = sc(p); y = np.r_[np.zeros(len(s_neg)), np.ones(len(s_pos))]
            r = dict(W=W, freq=f, banda='2.4' if f < 3000 else '5', distance_cm=dist, power_dbm=pw, n_cal=len(s_cal), n_neg=len(s_neg), n_pos=len(s_pos),
                     auroc=roc_auc_score(y, np.r_[s_neg, s_pos]))
            for a in (0.01, 0.001):
                h = conformal_threshold(s_cal, a); fp = int((s_neg > h).sum())
                r[f'tpr@{a:g}'] = float((s_pos > h).mean()); r[f'fpr@{a:g}'] = fp / len(s_neg); r[f'fpr_up95@{a:g}'] = fpr_upper95(fp, len(s_neg))
            rows.append(r)
S = pd.DataFrame(rows); S.to_csv(OUT / 'jamming_debole_stessa_campagna.csv', index=False)
print('negativi di test per canale (mediana):', int(S.n_neg.median()), '| calibrazione:', int(S.n_cal.median()))
for W in WINDOWS:
    print(f'\n=== W = {W}: AUROC medio sui canali ===')
    display(S[S.W == W].pivot_table(index=['banda', 'distance_cm'], columns='power_dbm', values='auroc', aggfunc='mean').round(3))
    print('rilevamento a α = 1% (FPR medio osservato:', round(S[S.W == W]['fpr@0.01'].mean(), 4), ')')
    display(S[S.W == W].pivot_table(index=['banda', 'distance_cm'], columns='power_dbm', values='tpr@0.01', aggfunc='mean').round(3))
print('\n=== canali più difficili a 60 cm, 0 dBm, W = 8 ===')
display(S[(S.W == 8) & (S.distance_cm == 60) & (S.power_dbm == 0)].sort_values('auroc')[['freq', 'auroc', 'tpr@0.01', 'fpr@0.01']].head(10).round(3))